<a href="https://colab.research.google.com/github/yt7360354-afk/vison_ai/blob/main/Week1_2_Convolution_Pooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 1주차 1-2. Convolution & Pooling 원리

> **시각지능(Vision AI) 고급 과정**
> **소요 시간:** 4시간 (이론 + 실습)
> **선수 지식:** 1-1 (CNN 패러다임과 이미지 표현) 완료

---

## 📚 학습 목표

이 실습을 마치면 여러분은 다음을 할 수 있게 됩니다.

1. **Convolution 연산**을 NumPy로 직접 구현하고, PyTorch 결과와 일치시킬 수 있다.
2. **Stride · Padding · Dilation**의 효과를 출력 크기 공식으로 예측·검증할 수 있다.
3. **Max/Avg/Global Pooling**의 차이를 시각적·수치적으로 설명할 수 있다.
4. **ReLU · GELU · Swish · Mish** 등 활성화 함수의 수식·미분·학습 영향을 비교할 수 있다.

## 🗂️ 차시 구성

| 부 | 주제 | 소요시간 |
|---|---|---|
| 1부 | Convolution 연산의 본질 (NumPy 직접 구현) | 60분 |
| 2부 | Stride · Padding · Dilation | 60분 |
| 3부 | Pooling과 Receptive Field | 60분 |
| 4부 | Activation Functions (ReLU·GELU·Swish·Mish) | 60분 |

## 💡 실습 환경

- Google Colab (GPU 권장)
- 패키지: `torch`, `torchvision`, `numpy`, `matplotlib`, `scipy`, `Pillow`


---
## 0. 환경 설정

In [ ]:
# 시드 고정 및 라이브러리 임포트
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import time

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 디바이스: {device}")
print(f"✅ PyTorch: {torch.__version__}")

# 한글 폰트 설정 (Colab)
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
import subprocess
try:
    subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
    print("✅ 한글 폰트 설정 완료")
except Exception as e:
    print(f"폰트 설정 생략: {e}")


---
# 🎬 1부. Convolution 연산의 본질 (60분)

## 🧠 이론: Convolution이란 무엇인가?

### 1) 합성곱(Convolution)의 수학적 정의

연속 시간에서:
$$ (f * g)(t) = \int_{-\infty}^{\infty} f(\tau) g(t-\tau) d\tau $$

이산 신호에서:
$$ (f * g)[n] = \sum_{m} f[m] \cdot g[n-m] $$

**딥러닝에서의 Convolution은 사실 Cross-correlation입니다.**
- 수학적 합성곱: 필터를 뒤집어서 곱
- 딥러닝 합성곱: **뒤집지 않고** 그대로 곱 (어차피 가중치를 학습하므로 무관)

### 2) 2D 이미지에서의 Convolution

$$ Y[i,j] = \sum_{m=0}^{K-1}\sum_{n=0}^{K-1} X[i+m, j+n] \cdot W[m,n] + b $$

- $X$: 입력 이미지
- $W$: 필터(커널), 크기 $K \times K$
- $Y$: 출력 특징맵(feature map)

### 3) 직관적 이해: 슬라이딩 윈도우

```
입력 (5×5)            필터 (3×3)        출력 (3×3)
┌─────────┐           ┌─────┐          ┌─────┐
│ 1 2 3 0 1│          │ 1 0 1│          │ ?  ? ?│
│ 0 1 2 3 0│    *     │ 0 1 0│    =     │ ?  ? ?│
│ 3 1 0 2 1│          │ 1 0 1│          │ ?  ? ?│
│ 2 0 1 3 2│          └─────┘          └─────┘
│ 1 2 3 0 1│
└─────────┘

좌상단 출력 = 1·1+2·0+3·1+0·0+1·1+2·0+3·1+1·0+0·1 = 8
```

### 4) 필터(커널)의 역할 = 특징 검출기

서로 다른 필터 = 서로 다른 특징을 검출

| 필터 | 효과 |
|---|---|
| `[[1,1,1],[1,1,1],[1,1,1]]/9` | 평균 블러 |
| `[[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]` | 에지 검출(라플라시안) |
| `[[-1,0,1],[-2,0,2],[-1,0,1]]` | 수직 에지 (Sobel-x) |
| `[[0,-1,0],[-1,5,-1],[0,-1,0]]` | 샤프닝 |

> 💡 **CNN의 핵심**: 이런 필터를 사람이 설계하지 않고, **학습으로 발견**합니다!

### 5) 다채널 Convolution

입력이 RGB 3채널이고, 출력 채널이 64개라면:
- 필터 한 개의 shape = `(3, K, K)` (입력 채널 수만큼 깊이)
- 64개 필터 전체 = `(64, 3, K, K)`
- PyTorch `Conv2d` 가중치: `(out_channels, in_channels, K_h, K_w)`

## 💻 실습 1-1. NumPy로 1D Convolution 직접 구현

먼저 가장 간단한 1차원에서 시작합니다.

In [ ]:
def conv1d_naive(x, w):
    """
    1D Convolution (실제로는 cross-correlation)
    x: 입력 신호 (length N)
    w: 필터 (length K)
    return: 출력 (length N-K+1)
    """
    N = len(x)
    K = len(w)
    out_len = N - K + 1
    y = np.zeros(out_len)
    for i in range(out_len):
        # x의 i번째부터 i+K번째까지를 w와 원소별 곱 후 합산
        y[i] = np.sum(x[i:i+K] * w)
    return y

# 테스트
x = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.float32)
w = np.array([1, 0, -1], dtype=np.float32)  # 미분 필터 (변화량 검출)

y = conv1d_naive(x, w)
print(f"입력 x: {x}")
print(f"필터 w: {w}")
print(f"출력 y: {y}")
print(f"출력 길이: {len(x)} - {len(w)} + 1 = {len(y)}")

# np.convolve와 비교 (np.convolve는 필터를 뒤집으므로 우리 결과와 다를 수 있음)
print(f"\nnp.convolve(mode='valid'): {np.convolve(x, w[::-1], mode='valid')}  ← 일치 확인")


## 💻 실습 1-2. NumPy로 2D Convolution 직접 구현 (Naive)

In [ ]:
def conv2d_naive(x, w):
    """
    2D Convolution (단일 채널, no padding, stride=1)
    x: (H, W) 입력
    w: (kH, kW) 커널
    return: (H-kH+1, W-kW+1) 출력
    """
    H, W = x.shape
    kH, kW = w.shape
    oH = H - kH + 1
    oW = W - kW + 1
    y = np.zeros((oH, oW), dtype=np.float32)

    for i in range(oH):
        for j in range(oW):
            # (i, j)부터 시작하는 (kH, kW) 영역과 커널의 원소별 곱 후 합산
            patch = x[i:i+kH, j:j+kW]
            y[i, j] = np.sum(patch * w)
    return y

# 검증: 작은 예시
x_small = np.array([
    [1, 2, 3, 0, 1],
    [0, 1, 2, 3, 0],
    [3, 1, 0, 2, 1],
    [2, 0, 1, 3, 2],
    [1, 2, 3, 0, 1]
], dtype=np.float32)

w_small = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
], dtype=np.float32)

y_small = conv2d_naive(x_small, w_small)
print(f"입력 shape: {x_small.shape}")
print(f"커널 shape: {w_small.shape}")
print(f"출력 shape: {y_small.shape}")
print(f"\n출력:\n{y_small}")
print(f"\n검증: 좌상단 = 1·1+2·0+3·1+0·0+1·1+2·0+3·1+1·0+0·1 = 8 → {y_small[0,0]}")


## 💻 실습 1-3. 다양한 커널 효과 시각화

직접 만든 `conv2d_naive`로 진짜 이미지에 다양한 필터를 적용해봅시다.

In [ ]:
# 샘플 이미지: scikit-image의 카메라맨 (없으면 자체 생성)
try:
    from skimage import data
    img = data.camera().astype(np.float32)  # (512, 512) 그레이스케일
except Exception:
    # scipy의 ascent 이미지로 대체
    from scipy.misc import ascent
    img = ascent().astype(np.float32)

print(f"이미지 shape: {img.shape}, range: [{img.min():.1f}, {img.max():.1f}]")
plt.figure(figsize=(5, 5))
plt.imshow(img, cmap='gray')
plt.title("원본 이미지")
plt.axis('off')
plt.show()


In [ ]:
# 다양한 커널 정의
kernels = {
    "Identity": np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float32),
    "평균 블러": np.ones((5,5), dtype=np.float32) / 25,
    "Sobel-X (수직 에지)": np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
    "Sobel-Y (수평 에지)": np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32),
    "Laplacian (전방향 에지)": np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=np.float32),
    "샤프닝": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, k) in zip(axes.flat, kernels.items()):
    out = conv2d_naive(img, k)
    ax.imshow(out, cmap='gray')
    ax.set_title(f"{name}\nshape: {k.shape}", fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 같은 입력에서 커널만 바꾸면 완전히 다른 특징이 추출됩니다.")
print("   CNN은 이런 커널 값을 사람이 정하지 않고 데이터에서 학습합니다!")


## 💻 실습 1-4. 빠른 구현: im2col 방식 (벡터화)

이중 for 문은 직관적이지만 매우 느립니다. **im2col** 트릭으로 행렬 곱셈으로 바꿔봅시다.

### im2col 아이디어
1. Convolution = 입력의 patch들과 커널의 dot product
2. 모든 patch를 **2D 행렬로 펼침** → 행렬 곱셈으로 한 번에 계산
3. GPU에서는 행렬 곱셈이 압도적으로 빠름

```
입력 (5×5)              im2col 변환 (9 patches × 9 elements)
┌─────────┐             ┌─────────────────┐
│ . . . . . │            │ patch1 (9 vals) │
│ . . . . . │     →      │ patch2 (9 vals) │  → 커널(9 vals)과 행렬 곱
│ . . . . . │            │  ...            │     = 출력 (9개 값)
│ . . . . . │            │ patch9 (9 vals) │
│ . . . . . │            └─────────────────┘
└─────────┘
```

In [ ]:
def im2col(x, kH, kW):
    """
    입력 (H, W)의 모든 패치를 (oH*oW, kH*kW) 행렬로 변환
    """
    H, W = x.shape
    oH, oW = H - kH + 1, W - kW + 1
    cols = np.zeros((oH * oW, kH * kW), dtype=x.dtype)
    for i in range(oH):
        for j in range(oW):
            patch = x[i:i+kH, j:j+kW].flatten()
            cols[i * oW + j] = patch
    return cols, oH, oW

def conv2d_im2col(x, w):
    """im2col + 행렬 곱셈으로 Convolution 수행"""
    kH, kW = w.shape
    cols, oH, oW = im2col(x, kH, kW)
    w_flat = w.flatten()           # (kH*kW,)
    y_flat = cols @ w_flat         # (oH*oW,)
    return y_flat.reshape(oH, oW)

# 정확성 검증: naive 결과와 동일한지
y_naive = conv2d_naive(x_small, w_small)
y_im2col = conv2d_im2col(x_small, w_small)
print(f"두 방식 결과 동일? {np.allclose(y_naive, y_im2col)}")


In [ ]:
# 속도 비교
import time

# 큰 이미지로 테스트
big_img = img  # 512×512
big_kernel = kernels["Sobel-X (수직 에지)"]

start = time.time()
y1 = conv2d_naive(big_img, big_kernel)
t_naive = time.time() - start

start = time.time()
y2 = conv2d_im2col(big_img, big_kernel)
t_im2col = time.time() - start

print(f"📊 512×512 이미지에 3×3 커널 적용 속도")
print(f"   Naive (이중 for문): {t_naive*1000:.1f} ms")
print(f"   im2col (행렬 곱):   {t_im2col*1000:.1f} ms")
print(f"   → im2col이 {t_naive/t_im2col:.1f}배 빠름")
print(f"\n   결과 동일? {np.allclose(y1, y2)}")
print()
print("💡 PyTorch/cuDNN의 Conv2d 내부도 결국 이런 행렬 곱 최적화를 사용합니다.")


## 💻 실습 1-5. PyTorch Conv2d와 결과 비교

직접 구현한 결과가 PyTorch와 일치하는지 검증합니다.

In [ ]:
# PyTorch Conv2d로 동일 연산
# Conv2d 입력: (N, C, H, W), 가중치: (out_C, in_C, kH, kW)

# 1) 입력 준비: (1, 1, H, W)
x_t = torch.from_numpy(big_img).unsqueeze(0).unsqueeze(0).float()

# 2) Conv 레이어 정의 (편향 없음)
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)

# 3) 우리가 만든 Sobel-X 커널을 가중치로 강제 설정
sobel_x = kernels["Sobel-X (수직 에지)"]
conv.weight.data = torch.from_numpy(sobel_x).unsqueeze(0).unsqueeze(0).float()

# 4) 추론
with torch.no_grad():
    y_torch = conv(x_t).squeeze().numpy()

print(f"우리 구현 출력 shape: {y2.shape}")
print(f"PyTorch 출력 shape: {y_torch.shape}")
print(f"두 결과 동일? {np.allclose(y2, y_torch, atol=1e-5)}")

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(big_img, cmap='gray'); axes[0].set_title("원본"); axes[0].axis('off')
axes[1].imshow(y2, cmap='gray'); axes[1].set_title("NumPy 직접 구현"); axes[1].axis('off')
axes[2].imshow(y_torch, cmap='gray'); axes[2].set_title("PyTorch Conv2d"); axes[2].axis('off')
plt.tight_layout(); plt.show()


## ✏️ 실습 과제 1-A

**문제:** 다음 가우시안 커널을 만들고 이미지에 적용해보세요.

가우시안 커널 5×5 (σ=1.0):
$$ G(i,j) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{(i-c)^2 + (j-c)^2}{2\sigma^2}\right) $$

여기서 $c$는 중앙 인덱스(=2)입니다. 합이 1이 되도록 정규화하세요.

In [ ]:
# TODO: 가우시안 커널 생성 함수 작성
def gaussian_kernel(size, sigma):
    """size×size 가우시안 커널 생성"""
    # 힌트: np.indices로 좌표 만들고, 중앙(c)으로부터 거리 계산
    c = size // 2
    # ii, jj = np.indices((size, size))
    # kernel = np.exp(-((ii - c)**2 + (jj - c)**2) / (2 * sigma**2))
    # kernel = kernel / kernel.sum()  # 합=1로 정규화
    pass  # 여기를 채우세요

# 실행
# gauss = gaussian_kernel(5, 1.0)
# print("가우시안 커널 (5×5, σ=1.0):")
# print(gauss)
# print(f"합: {gauss.sum():.4f}  ← 1.0이어야 함")

# blurred = conv2d_im2col(big_img, gauss)
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# axes[0].imshow(big_img, cmap='gray'); axes[0].set_title("원본"); axes[0].axis('off')
# axes[1].imshow(blurred, cmap='gray'); axes[1].set_title("가우시안 블러"); axes[1].axis('off')
# plt.show()


<details>
<summary>🔑 정답 보기</summary>

```python
def gaussian_kernel(size, sigma):
    c = size // 2
    ii, jj = np.indices((size, size))
    kernel = np.exp(-((ii - c)**2 + (jj - c)**2) / (2 * sigma**2))
    return kernel / kernel.sum()

gauss = gaussian_kernel(5, 1.0)
blurred = conv2d_im2col(big_img, gauss)
```
</details>

---
# 🎬 2부. Stride · Padding · Dilation (60분)

## 🧠 이론: 출력 크기를 제어하는 3대 파라미터

### 1) Stride (보폭)

필터를 한 번에 **몇 칸 이동**할지 결정.
- `stride=1`: 한 칸씩 (기본값)
- `stride=2`: 두 칸씩 (출력 크기 절반으로 다운샘플링)

```
stride=1 (출력 5×5)         stride=2 (출력 3×3)
□□□ . .                    □□□ . .
□□□ . . → ↓                . . . . .  → ↓↓
□□□ . .                    □□□ . .
. . . . .                  . . . . .
. . . . .                  □□□ . .
```

### 2) Padding (테두리)

입력 가장자리에 **추가 픽셀 채우기**. 가장자리 정보 손실 방지.

| 종류 | 설명 |
|---|---|
| `valid` | 패딩 없음 → 출력 크기 감소 |
| `same` | 출력 크기 = 입력 크기 (자동 padding) |
| `zero` | 0으로 채움 (가장 일반적) |
| `reflect` | 거울 반사 |
| `replicate` | 가장자리 복제 |

### 3) Dilation (팽창)

필터의 **원소 사이 간격**. Receptive field 확장에 효과적.

```
dilation=1 (보통)        dilation=2 (구멍 뚫린 필터)
□ □ □                    □ . □ . □
□ □ □                    . . . . .
□ □ □                    □ . □ . □
                         . . . . .
                         □ . □ . □
3×3 영역 커버              5×5 영역 커버 (파라미터는 그대로!)
```

### 4) 출력 크기 공식 (★암기)

$$ O = \left\lfloor \frac{W - K_{\text{eff}} + 2P}{S} \right\rfloor + 1 $$

- $W$: 입력 크기
- $K$: 커널 크기
- $K_{\text{eff}} = D \cdot (K-1) + 1$: dilation 적용된 유효 커널 크기
- $P$: padding
- $S$: stride

### 5) "Same Padding" 계산법

출력 = 입력으로 만들려면? (stride=1, dilation=1 가정)
$$ P = \frac{K - 1}{2} $$

- K=3 → P=1
- K=5 → P=2
- K=7 → P=3

## 💻 실습 2-1. Stride 효과 시각화

In [ ]:
# PyTorch Conv2d로 stride 비교
def make_conv_with_kernel(kernel_np, stride=1, padding=0, dilation=1):
    """NumPy 커널을 가진 Conv2d 레이어 생성"""
    kH, kW = kernel_np.shape
    conv = nn.Conv2d(1, 1, kernel_size=(kH, kW),
                     stride=stride, padding=padding,
                     dilation=dilation, bias=False)
    conv.weight.data = torch.from_numpy(kernel_np).unsqueeze(0).unsqueeze(0).float()
    return conv

x_t = torch.from_numpy(big_img).unsqueeze(0).unsqueeze(0).float()
sobel_x = kernels["Sobel-X (수직 에지)"]

# 다양한 stride
strides = [1, 2, 3, 4]
fig, axes = plt.subplots(1, len(strides), figsize=(16, 4))

for ax, s in zip(axes, strides):
    conv = make_conv_with_kernel(sobel_x, stride=s)
    with torch.no_grad():
        y = conv(x_t).squeeze().numpy()
    ax.imshow(y, cmap='gray')
    ax.set_title(f"stride={s}\n출력: {y.shape}")
    ax.axis('off')
plt.tight_layout(); plt.show()

# 출력 크기 공식 검증
W = 512; K = 3; P = 0; D = 1
print("\n📐 출력 크기 공식 검증 (입력=512, K=3, P=0)")
for s in strides:
    K_eff = D * (K - 1) + 1
    O = (W - K_eff + 2*P) // s + 1
    print(f"   stride={s}: 공식 결과 = {O}")


## 💻 실습 2-2. Padding 효과 비교

In [ ]:
# 패딩 종류별 비교
small_img = big_img[:200, :200]  # 작게 잘라서 가장자리 효과 잘 보이게
x_t_small = torch.from_numpy(small_img).unsqueeze(0).unsqueeze(0).float()

paddings = [0, 1, 3, 10]
fig, axes = plt.subplots(1, len(paddings), figsize=(16, 4))

for ax, p in zip(axes, paddings):
    conv = make_conv_with_kernel(sobel_x, stride=1, padding=p)
    with torch.no_grad():
        y = conv(x_t_small).squeeze().numpy()
    ax.imshow(y, cmap='gray')
    ax.set_title(f"padding={p}\n출력: {y.shape}")
    ax.axis('off')
plt.tight_layout(); plt.show()

print("💡 padding이 클수록 가장자리에 검은 띠(0으로 패딩된 영역)가 보입니다.")


In [ ]:
# Padding 모드 비교: constant(zero) vs reflect vs replicate
modes = {
    "constant (0으로 채움)": "constant",   # ← "zeros"가 아닌 "constant"
    "reflect (거울 반사)": "reflect",
    "replicate (가장자리 복제)": "replicate",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
tiny_img = big_img[:50, :50]  # 매우 작게 → padding 효과 잘 보임
x_tiny = torch.from_numpy(tiny_img).unsqueeze(0).unsqueeze(0).float()

for ax, (name, mode) in zip(axes, modes.items()):
    # F.pad로 다양한 모드 적용
    padded = F.pad(x_tiny, (10, 10, 10, 10), mode=mode)
    ax.imshow(padded.squeeze().numpy(), cmap='gray')
    ax.set_title(f"{name}\n원본 50×50 → {tuple(padded.shape[-2:])}")
    ax.axis('off')
    # 원본 영역 표시
    rect = Rectangle((10, 10), 50, 50, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
plt.suptitle("패딩 모드별 차이 (빨간 박스 = 원본 영역)", y=1.02)
plt.tight_layout(); plt.show()

## 💻 실습 2-3. Dilation: Receptive Field를 키우는 트릭

In [ ]:
# Dilation 시각화: 같은 3×3 커널이 dilation에 따라 얼마나 큰 영역을 보는지
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, d in zip(axes, [1, 2, 3, 4]):
    K = 3
    K_eff = d * (K - 1) + 1
    grid_size = K_eff + 2

    # 시각화용 그리드
    grid = np.zeros((grid_size, grid_size))
    center = grid_size // 2

    # 커널이 닿는 위치들 표시
    for i in range(-(K//2), K//2 + 1):
        for j in range(-(K//2), K//2 + 1):
            grid[center + i*d, center + j*d] = 1

    ax.imshow(grid, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"dilation={d}\n유효 커널: {K_eff}×{K_eff}")
    ax.set_xticks([]); ax.set_yticks([])
    # 그리드 선 그리기
    for i in range(grid_size + 1):
        ax.axhline(i - 0.5, color='gray', linewidth=0.5)
        ax.axvline(i - 0.5, color='gray', linewidth=0.5)

plt.suptitle("Dilation: 파라미터는 9개로 동일하지만 보는 영역이 커진다", y=1.02)
plt.tight_layout(); plt.show()

print("💡 Dilated Convolution의 활용:")
print("   - 세그멘테이션(DeepLab) - 해상도 유지하며 receptive field 확대")
print("   - WaveNet - 오디오 long-range dependency")


In [ ]:
# 실제 이미지에 dilation 적용
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, d in zip(axes, [1, 2, 4, 8]):
    # padding을 dilation에 맞춰 same-size 유지
    p = d  # K=3, padding=dilation이면 same size
    conv = make_conv_with_kernel(sobel_x, stride=1, padding=p, dilation=d)
    with torch.no_grad():
        y = conv(x_t).squeeze().numpy()
    ax.imshow(y, cmap='gray')
    K_eff = d * (3 - 1) + 1
    ax.set_title(f"dilation={d}\n유효 커널: {K_eff}×{K_eff}")
    ax.axis('off')
plt.tight_layout(); plt.show()

print("💡 같은 3×3 필터(파라미터 9개)지만 dilation이 커질수록 더 멀리 보고 있습니다.")


## 💻 실습 2-4. 출력 크기 계산 연습 - 직접 검증

In [ ]:
def compute_output_size(W, K, S, P, D=1):
    """Conv2d 출력 크기 계산"""
    K_eff = D * (K - 1) + 1
    return (W - K_eff + 2*P) // S + 1

# 다양한 조합으로 검증
test_cases = [
    # (W, K, S, P, D)
    (32, 3, 1, 0, 1),
    (32, 3, 1, 1, 1),    # same padding
    (32, 5, 1, 2, 1),    # same padding
    (32, 3, 2, 1, 1),    # 다운샘플
    (224, 7, 2, 3, 1),   # ResNet 첫 층
    (224, 3, 1, 2, 2),   # dilation
]

print(f"{'입력':>6} | {'K':>2} | {'S':>2} | {'P':>2} | {'D':>2} | {'공식':>4} | {'PyTorch':>7}")
print("-" * 50)

for W, K, S, P, D in test_cases:
    # 공식 결과
    formula = compute_output_size(W, K, S, P, D)

    # PyTorch 실제 결과
    x = torch.zeros(1, 1, W, W)
    conv = nn.Conv2d(1, 1, kernel_size=K, stride=S, padding=P, dilation=D, bias=False)
    with torch.no_grad():
        y = conv(x)
    pytorch_size = y.shape[-1]

    match = "✅" if formula == pytorch_size else "❌"
    print(f"{W:>6} | {K:>2} | {S:>2} | {P:>2} | {D:>2} | {formula:>4} | {pytorch_size:>7} {match}")


## ✏️ 실습 과제 2-A

**문제:** 입력 224×224 이미지에 다음 Conv 레이어를 통과시키면 출력 크기는?

| 레이어 | K | S | P | D |
|---|---|---|---|---|
| Conv1 | 7 | 2 | 3 | 1 |
| Conv2 | 3 | 1 | 1 | 1 |
| Conv3 | 3 | 2 | 1 | 1 |
| Conv4 | 3 | 1 | 2 | 2 |

각 레이어 출력 크기를 공식으로 계산한 뒤 PyTorch로 검증하세요.

In [ ]:
# TODO: 각 레이어 출력 크기 계산
W = 224

# layers = [(7, 2, 3, 1), (3, 1, 1, 1), (3, 2, 1, 1), (3, 1, 2, 2)]
# for i, (K, S, P, D) in enumerate(layers, 1):
#     W = compute_output_size(W, K, S, P, D)
#     print(f"Conv{i} 후: {W}×{W}")

# 검증: 실제 모델 만들기
# model = nn.Sequential(
#     nn.Conv2d(1, 1, kernel_size=7, stride=2, padding=3, bias=False),
#     nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False),
#     nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1, bias=False),
#     nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=2, dilation=2, bias=False),
# )
# x = torch.zeros(1, 1, 224, 224)
# print(f"\n실제 출력: {model(x).shape}")


<details>
<summary>🔑 정답 보기</summary>

| 레이어 | 출력 크기 |
|---|---|
| Conv1 (K=7, S=2, P=3) | (224-7+6)/2 + 1 = **112** |
| Conv2 (K=3, S=1, P=1) | (112-3+2)/1 + 1 = **112** |
| Conv3 (K=3, S=2, P=1) | (112-3+2)/2 + 1 = **56** |
| Conv4 (K=3, S=1, P=2, D=2) | K_eff=5, (56-5+4)/1 + 1 = **56** |

</details>

---
# 🎬 3부. Pooling과 Receptive Field (60분)

## 🧠 이론: Pooling의 역할

### 1) Pooling이란?

**다운샘플링(downsampling)** 연산. 학습 파라미터 없음.
- 입력의 일정 영역을 **하나의 값으로 요약**
- 가장 흔한 형태: 2×2 윈도우, stride=2 → 크기 절반

### 2) Pooling의 3가지 효과

✅ **공간 정보 압축**: 해상도 감소 → 연산량 감소
✅ **위치 불변성 강화**: 작은 위치 변화에 둔감
✅ **Receptive Field 확장**: 깊은 층이 더 넓은 영역을 보게 됨

### 3) 종류

| 종류 | 수식 | 특징 |
|---|---|---|
| **Max Pooling** | $y = \max(x_{i,j})$ | 강한 특징(에지, 코너) 보존 |
| **Average Pooling** | $y = \frac{1}{n}\sum x_{i,j}$ | 부드러운 다운샘플링 |
| **Global Avg Pooling** | 채널별 전체 평균 → 1개 값 | 분류기 직전, FC 대체 |
| **Adaptive Pooling** | 출력 크기 지정 | 입력 크기 가변 모델 |

### 4) Strided Convolution vs Pooling

같은 다운샘플링이지만 차이가 있습니다.

| 기법 | 학습 파라미터 | 추가 연산 | 사용처 |
|---|---|---|---|
| MaxPool 2×2, s=2 | 없음 | 거의 없음 | 전통 CNN (VGG, ResNet 초기) |
| Conv 3×3, s=2 | 있음 | Conv 1번 | 최신 CNN (EfficientNet) |

> 💡 **트렌드**: 최근 모델은 풀링 대신 Strided Conv를 선호 (학습 가능 다운샘플링)

### 5) Receptive Field (수용 영역)

**한 출력 픽셀이 입력의 얼마나 큰 영역에서 정보를 받는가?**

각 층을 지날 때마다 receptive field가 커집니다.

층마다 누적 계산:
$$ RF_{l} = RF_{l-1} + (K_l - 1) \cdot \prod_{i=1}^{l-1} S_i $$

**예시: Conv3-Pool2-Conv3 적용 시**

| 층 | 연산 | RF |
|---|---|---|
| 입력 | - | 1 |
| Conv 3×3 | RF=1+(3-1)·1 | **3** |
| MaxPool 2×2 | RF=3+(2-1)·1 | **4** |
| Conv 3×3 | RF=4+(3-1)·2 | **8** |

깊을수록 한 픽셀이 입력의 넓은 영역을 본다 = "전체 맥락" 파악 가능.

## 💻 실습 3-1. Max/Avg Pooling을 NumPy로 직접 구현

In [ ]:
def pool2d(x, kernel_size, stride=None, mode='max'):
    """
    2D Pooling 직접 구현
    x: (H, W) 입력
    kernel_size: pool 윈도우 크기
    stride: 보폭 (기본 = kernel_size)
    mode: 'max' or 'avg'
    """
    if stride is None:
        stride = kernel_size
    H, W = x.shape
    oH = (H - kernel_size) // stride + 1
    oW = (W - kernel_size) // stride + 1
    y = np.zeros((oH, oW), dtype=x.dtype)

    for i in range(oH):
        for j in range(oW):
            patch = x[i*stride : i*stride + kernel_size,
                      j*stride : j*stride + kernel_size]
            if mode == 'max':
                y[i, j] = patch.max()
            elif mode == 'avg':
                y[i, j] = patch.mean()
            else:
                raise ValueError("mode는 'max' 또는 'avg'")
    return y

# 작은 예제로 검증
test_input = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 2, 4, 1],
    [1, 0, 3, 2]
], dtype=np.float32)

print("입력:\n", test_input)
print("\nMax Pool 2×2, stride 2:\n", pool2d(test_input, 2, 2, mode='max'))
print("\nAvg Pool 2×2, stride 2:\n", pool2d(test_input, 2, 2, mode='avg'))


In [ ]:
# PyTorch와 결과 비교
x_t = torch.from_numpy(test_input).unsqueeze(0).unsqueeze(0)

torch_max = F.max_pool2d(x_t, 2, 2).squeeze().numpy()
torch_avg = F.avg_pool2d(x_t, 2, 2).squeeze().numpy()

ours_max = pool2d(test_input, 2, 2, 'max')
ours_avg = pool2d(test_input, 2, 2, 'avg')

print(f"Max Pool 일치? {np.allclose(ours_max, torch_max)}")
print(f"Avg Pool 일치? {np.allclose(ours_avg, torch_avg)}")


## 💻 실습 3-2. Pooling 시각적 비교

In [ ]:
# 실제 이미지에 다양한 풀링 적용
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Max Pooling 행
for ax, k in zip(axes[0], [2, 4, 8, 16]):
    pooled = pool2d(big_img, k, k, mode='max')
    ax.imshow(pooled, cmap='gray')
    ax.set_title(f"Max Pool {k}×{k}\n출력: {pooled.shape}")
    ax.axis('off')

# Avg Pooling 행
for ax, k in zip(axes[1], [2, 4, 8, 16]):
    pooled = pool2d(big_img, k, k, mode='avg')
    ax.imshow(pooled, cmap='gray')
    ax.set_title(f"Avg Pool {k}×{k}\n출력: {pooled.shape}")
    ax.axis('off')

axes[0, 0].set_ylabel("Max Pool", fontsize=14, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Avg Pool", fontsize=14, rotation=0, labelpad=40)
plt.tight_layout(); plt.show()

print("💡 관찰:")
print("   - Max Pool: 강한 특징(밝은 픽셀)이 살아남아 'noisy'하지만 디테일 보존")
print("   - Avg Pool: 부드럽게 다운샘플링, 블러 효과 유사")


## 💻 실습 3-3. Strided Conv vs Max Pool 비교

같은 4× 다운샘플링을 두 방식으로 수행.

In [ ]:
x_t = torch.from_numpy(big_img).unsqueeze(0).unsqueeze(0).float()

# 방식 1: MaxPool 4×4 (학습 파라미터 0개)
maxpool = nn.MaxPool2d(kernel_size=4, stride=4)
y_pool = maxpool(x_t).squeeze().numpy()

# 방식 2: Strided Conv 4×4 stride 4 (학습 파라미터 16개)
conv_strided = nn.Conv2d(1, 1, kernel_size=4, stride=4, bias=False)
# 여기서는 학습 안 했으니 임의 가중치
with torch.no_grad():
    y_conv = conv_strided(x_t).squeeze().numpy()

# 방식 3: Avg Pool (참고)
y_avg = F.avg_pool2d(x_t, 4, 4).squeeze().numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(big_img, cmap='gray'); axes[0].set_title(f"원본 {big_img.shape}"); axes[0].axis('off')
axes[1].imshow(y_pool, cmap='gray'); axes[1].set_title(f"MaxPool 4×4\n파라미터 0개"); axes[1].axis('off')
axes[2].imshow(y_avg, cmap='gray'); axes[2].set_title(f"AvgPool 4×4\n파라미터 0개"); axes[2].axis('off')
axes[3].imshow(y_conv, cmap='gray'); axes[3].set_title(f"Conv 4×4 s=4\n파라미터 16개 (학습 가능)"); axes[3].axis('off')
plt.tight_layout(); plt.show()

print("💡 같은 다운샘플링이지만:")
print("   - Pool: 고정된 규칙 (max/avg)")
print("   - Strided Conv: 데이터에 맞게 다운샘플링 패턴을 학습")


## 💻 실습 3-4. Global Average Pooling (GAP)

최근 분류 모델은 Flatten + FC 대신 **Global Average Pooling**을 자주 씁니다.

In [ ]:
# 가짜 feature map (32 채널, 7×7 공간)
feature_map = torch.randn(1, 32, 7, 7)
print(f"Feature map: {feature_map.shape}")

# 방식 1: 전통적 - Flatten + FC
flatten = nn.Flatten()
fc = nn.Linear(32 * 7 * 7, 10)
out_traditional = fc(flatten(feature_map))
print(f"\n[전통적] Flatten + FC")
print(f"   Flatten 결과: {flatten(feature_map).shape}")
print(f"   FC 파라미터: {32*7*7*10 + 10:,}개")
print(f"   출력: {out_traditional.shape}")

# 방식 2: GAP + FC (현대적)
gap = nn.AdaptiveAvgPool2d(1)              # 출력을 (1, 1)로
gap_out = gap(feature_map)                  # (1, 32, 1, 1)
gap_flat = gap_out.view(1, -1)              # (1, 32)
fc_gap = nn.Linear(32, 10)
out_gap = fc_gap(gap_flat)
print(f"\n[GAP] Global Avg Pool + FC")
print(f"   GAP 결과: {gap_out.shape} → flatten {gap_flat.shape}")
print(f"   FC 파라미터: {32*10 + 10:,}개")
print(f"   출력: {out_gap.shape}")

print(f"\n💡 파라미터 수: 전통적 {32*7*7*10+10:,} vs GAP {32*10+10:,}")
print(f"   {(32*7*7*10+10)/(32*10+10):.1f}배 적은 파라미터로 동일한 분류기 구현!")


## 💻 실습 3-5. Receptive Field 직접 계산

In [ ]:
def compute_receptive_field(layers):
    """
    layers: [(K, S), (K, S), ...] 형태의 레이어 리스트
    각 층까지의 누적 receptive field 계산
    """
    rf = 1            # 입력의 한 픽셀
    jump = 1          # 누적 stride

    print(f"{'층':>6} | {'K':>2} | {'S':>2} | {'RF':>4} | {'jump':>4}")
    print("-" * 40)
    print(f"{'입력':>6} |    |    | {rf:>4} | {jump:>4}")

    for i, (K, S) in enumerate(layers, 1):
        rf = rf + (K - 1) * jump
        jump = jump * S
        print(f"{'L'+str(i):>6} | {K:>2} | {S:>2} | {rf:>4} | {jump:>4}")
    return rf

# VGG 스타일 미니 네트워크
print("📐 VGG 스타일 (Conv3-Conv3-Pool2-Conv3-Conv3-Pool2-Conv3-Conv3)")
layers = [
    (3, 1),  # Conv 3×3
    (3, 1),  # Conv 3×3
    (2, 2),  # Pool 2×2
    (3, 1),  # Conv 3×3
    (3, 1),  # Conv 3×3
    (2, 2),  # Pool 2×2
    (3, 1),  # Conv 3×3
    (3, 1),  # Conv 3×3
]
final_rf = compute_receptive_field(layers)
print(f"\n→ 최종 한 픽셀이 입력의 {final_rf}×{final_rf} 영역을 본다!")


In [ ]:
# Receptive Field 시각화: 깊이에 따라 한 출력 픽셀이 보는 영역
def visualize_rf_growth():
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    rf_history = [1, 3, 7, 14]  # 단순화
    titles = ["입력", "Conv1 후", "Conv2 후 (+Pool)", "Conv3 후"]

    for ax, rf, title in zip(axes, rf_history, titles):
        canvas = np.ones((30, 30, 3))  # 흰 캔버스

        # RF 영역을 빨간 사각형으로
        center = 15
        half = rf // 2
        y0, y1 = max(0, center-half), min(30, center+half+1)
        x0, x1 = max(0, center-half), min(30, center+half+1)
        canvas[y0:y1, x0:x1] = [1, 0.7, 0.7]

        ax.imshow(canvas)
        # 격자
        for i in range(31):
            ax.axhline(i - 0.5, color='gray', linewidth=0.3)
            ax.axvline(i - 0.5, color='gray', linewidth=0.3)
        ax.set_title(f"{title}\nRF = {rf}×{rf}")
        ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle("층이 깊어질수록 한 픽셀이 보는 영역(빨간 영역)이 커진다", y=1.02)
    plt.tight_layout(); plt.show()

visualize_rf_growth()
print("💡 깊은 층의 한 뉴런 = 입력 이미지의 큰 영역을 종합한 추상적 특징")
print("   얕은 층: 에지·텍스처 / 깊은 층: 객체 부분 → 객체 전체")


## ✏️ 실습 과제 3-A

**문제:** 다음 네트워크의 최종 receptive field를 계산하세요.

| 층 | 종류 | K | S |
|---|---|---|---|
| 1 | Conv | 7 | 2 |
| 2 | Pool | 3 | 2 |
| 3 | Conv | 3 | 1 |
| 4 | Conv | 3 | 1 |
| 5 | Pool | 2 | 2 |
| 6 | Conv | 3 | 1 |

In [ ]:
# TODO: 위 표를 (K, S) 리스트로 만들고 compute_receptive_field로 계산
# layers = [(7, 2), (3, 2), ...]
# rf = compute_receptive_field(layers)


<details>
<summary>🔑 정답 보기</summary>

```python
layers = [(7, 2), (3, 2), (3, 1), (3, 1), (2, 2), (3, 1)]
rf = compute_receptive_field(layers)  # → 35
```

| 층 | K | S | RF | jump |
|---|---|---|---|---|
| 입력 |  |  | 1 | 1 |
| Conv 7×7 s=2 | 7 | 2 | 1+6·1=7 | 2 |
| Pool 3×3 s=2 | 3 | 2 | 7+2·2=11 | 4 |
| Conv 3×3 s=1 | 3 | 1 | 11+2·4=19 | 4 |
| Conv 3×3 s=1 | 3 | 1 | 19+2·4=27 | 4 |
| Pool 2×2 s=2 | 2 | 2 | 27+1·4=31 | 8 |
| Conv 3×3 s=1 | 3 | 1 | 31+2·8=**47** | 8 |

</details>

---
# 🎬 4부. Activation Functions (60분)

## 🧠 이론: 비선형성의 힘

### 1) 왜 활성화 함수가 필요한가?

**선형 변환의 합성은 여전히 선형입니다.**

$$ f(x) = W_2(W_1 x + b_1) + b_2 = (W_2 W_1) x + (W_2 b_1 + b_2) $$

활성화 함수가 없으면 100층 네트워크도 1층 네트워크와 동일!

→ **비선형 활성화 함수 = 신경망의 표현력의 원천**

### 2) 전통적 활성화 함수

#### Sigmoid: $\sigma(x) = \frac{1}{1+e^{-x}}$
- 출력 범위: (0, 1)
- ❌ Vanishing Gradient 문제 (양 끝에서 미분 ≈ 0)
- ❌ 출력이 0 중심이 아님

#### Tanh: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
- 출력 범위: (-1, 1)
- ✅ 0 중심 → Sigmoid보다 빠른 학습
- ❌ 여전히 Vanishing Gradient

#### ReLU: $\text{ReLU}(x) = \max(0, x)$
- ✅ 계산 매우 빠름 (단순 비교)
- ✅ 양수 영역에서 그래디언트 = 1 (Vanishing 없음)
- ❌ **Dying ReLU**: 음수에서 영구 0 → 일부 뉴런 사망

### 3) ReLU 변형들

#### Leaky ReLU: $\max(0.01x, x)$
- 음수에 작은 기울기 부여 → Dying ReLU 완화

#### ELU: $x$ if $x>0$, $\alpha(e^x - 1)$ if $x \le 0$
- 음수에서 부드러운 곡선

### 4) 현대적 활성화 함수

#### GELU: $x \cdot \Phi(x)$
- $\Phi$: 표준정규 누적분포함수
- 근사: $0.5 x (1 + \tanh(\sqrt{2/\pi}(x + 0.044715 x^3)))$
- ✅ 부드럽고 0 근처에서 자연스러운 게이팅
- 🚀 **BERT, GPT, ViT의 표준**

#### Swish (SiLU): $x \cdot \sigma(x)$
- ✅ 단순하면서 GELU와 유사한 성능
- 🚀 **EfficientNet, YOLOv5+의 표준**

#### Mish: $x \cdot \tanh(\ln(1 + e^x))$
- ✅ 더 부드러움
- 🚀 **YOLOv4의 표준**

### 5) 한 줄 요약

| 모델 | 활성화 함수 |
|---|---|
| 1990s LeNet | Sigmoid/Tanh |
| 2012 AlexNet | **ReLU** (혁명) |
| 2015 ResNet | ReLU |
| 2017 Transformer | ReLU |
| 2018 BERT/GPT | **GELU** |
| 2019 EfficientNet | **Swish/SiLU** |
| 2020 YOLOv4 | **Mish** |
| 2021+ ViT, ConvNeXt | **GELU** |

## 💻 실습 4-1. 모든 활성화 함수 시각화 (함수 + 미분)

In [ ]:
# NumPy로 직접 구현
def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh(x): return np.tanh(x)
def relu(x): return np.maximum(0, x)
def leaky_relu(x, alpha=0.01): return np.where(x > 0, x, alpha * x)
def elu(x, alpha=1.0): return np.where(x > 0, x, alpha * (np.exp(x) - 1))
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
def swish(x): return x * sigmoid(x)
def mish(x): return x * np.tanh(np.log1p(np.exp(x)))

# 미분
def d_sigmoid(x): s = sigmoid(x); return s * (1 - s)
def d_tanh(x): return 1 - np.tanh(x)**2
def d_relu(x): return (x > 0).astype(np.float32)
def d_leaky_relu(x, alpha=0.01): return np.where(x > 0, 1, alpha)
def d_elu(x, alpha=1.0): return np.where(x > 0, 1, alpha * np.exp(x))

# GELU/Swish/Mish 미분은 복잡하니 PyTorch autograd 사용
def get_pytorch_grad(act_fn, x_np):
    x = torch.tensor(x_np, dtype=torch.float32, requires_grad=True)
    y = act_fn(x)
    y.sum().backward()
    return x.grad.numpy()

x = np.linspace(-5, 5, 200)

functions = {
    'Sigmoid': (sigmoid(x), d_sigmoid(x)),
    'Tanh': (tanh(x), d_tanh(x)),
    'ReLU': (relu(x), d_relu(x)),
    'Leaky ReLU': (leaky_relu(x), d_leaky_relu(x)),
    'ELU': (elu(x), d_elu(x)),
    'GELU': (gelu(x), get_pytorch_grad(F.gelu, x)),
    'Swish (SiLU)': (swish(x), get_pytorch_grad(F.silu, x)),
    'Mish': (mish(x), get_pytorch_grad(F.mish, x)),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (name, (y, dy)) in zip(axes.flat, functions.items()):
    ax.plot(x, y, 'b-', linewidth=2, label=name)
    ax.plot(x, dy, 'r--', linewidth=2, label=f'미분')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=12)
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(-1.5, 5)
plt.tight_layout(); plt.show()


## 💻 실습 4-2. 한 그림에 모두 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 함수 비교
for name, (y, dy) in functions.items():
    axes[0].plot(x, y, label=name, linewidth=2)
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].set_title("활성화 함수 비교")
axes[0].set_xlabel("입력 x")
axes[0].set_ylabel("출력 f(x)")
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(-2, 5)

# 미분 비교
for name, (y, dy) in functions.items():
    axes[1].plot(x, dy, label=name, linewidth=2)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].axvline(0, color='gray', linewidth=0.5)
axes[1].set_title("미분(그래디언트) 비교")
axes[1].set_xlabel("입력 x")
axes[1].set_ylabel("f'(x)")
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(-0.5, 2)

plt.tight_layout(); plt.show()

print("💡 핵심 관찰:")
print("   - Sigmoid/Tanh: 양 끝에서 미분 ≈ 0 (Vanishing Gradient)")
print("   - ReLU: 음수에서 미분 = 0 (Dying ReLU)")
print("   - GELU/Swish/Mish: 음수 영역에도 작은 그래디언트 → 부드러운 학습")


## 💻 실습 4-3. Dying ReLU 문제 시뮬레이션

In [ ]:
# 시나리오: 가중치 초기화가 잘못되어 모든 입력이 음수가 되면?
torch.manual_seed(0)
n_neurons = 100
n_samples = 500

# 입력 데이터
data = torch.randn(n_samples, 10)

# 의도적으로 큰 음수 편향 → ReLU가 0을 반환할 가능성 높음
linear = nn.Linear(10, n_neurons)
linear.bias.data = torch.full((n_neurons,), -3.0)  # 큰 음수 편향

# ⚠️ no_grad로 감싸야 .numpy() 직접 호출 가능 (gradient 추적 불필요)
with torch.no_grad():
    pre_activation = linear(data)        # (500, 100)
    relu_out = F.relu(pre_activation)
    gelu_out = F.gelu(pre_activation)
    swish_out = F.silu(pre_activation)

# "죽은 뉴런" = 모든 입력에 대해 0인 뉴런
dead_relu = ((relu_out == 0).all(dim=0)).sum().item()
near_dead_gelu = ((gelu_out.abs() < 0.001).all(dim=0)).sum().item()
near_dead_swish = ((swish_out.abs() < 0.001).all(dim=0)).sum().item()

print(f"📊 100개 뉴런 중 '거의 죽은' 뉴런 개수")
print(f"   ReLU:  {dead_relu}개 (완전 사망)")
print(f"   GELU:  {near_dead_gelu}개")
print(f"   Swish: {near_dead_swish}개")

# 출력 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(relu_out.flatten().numpy(), bins=50, color='red', alpha=0.7)
axes[0].set_title(f"ReLU 출력 분포\n0 비율: {(relu_out == 0).float().mean():.2%}")
axes[0].set_xlabel("출력값")

axes[1].hist(gelu_out.flatten().numpy(), bins=50, color='green', alpha=0.7)
axes[1].set_title(f"GELU 출력 분포\n매우 작은 값 비율: {(gelu_out.abs() < 0.01).float().mean():.2%}")
axes[1].set_xlabel("출력값")

axes[2].hist(swish_out.flatten().numpy(), bins=50, color='blue', alpha=0.7)
axes[2].set_title(f"Swish 출력 분포\n매우 작은 값 비율: {(swish_out.abs() < 0.01).float().mean():.2%}")
axes[2].set_xlabel("출력값")
plt.tight_layout(); plt.show()

In [ ]:
# 그래디언트 비교: 입력 x=-3에서 각 활성화의 미분값
x_test = torch.tensor([-3.0], requires_grad=True)

for name, fn in [('ReLU', F.relu), ('GELU', F.gelu), ('Swish', F.silu)]:
    x = torch.tensor([-3.0], requires_grad=True)
    fn(x).backward()
    print(f"{name:>6} gradient at x=-3: {x.grad.item():.6f}")

## 💻 실습 4-4. 활성화 함수별 학습 성능 비교 (MNIST)

In [ ]:
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = MNIST('./data', train=True, download=True, transform=mnist_transform)
mnist_test = MNIST('./data', train=False, download=True, transform=mnist_transform)

train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(mnist_test, batch_size=256, shuffle=False, num_workers=2)


In [ ]:
# 동일한 CNN 구조에 활성화만 다르게 → 성능 비교
class CNN_with_act(nn.Module):
    def __init__(self, act_fn):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(32*7*7, 10)
        self.act = act_fn

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = x.flatten(1)
        return self.fc(x)

def train_one_epoch(model, loader, opt, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        opt.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        opt.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model(imgs)
        loss = criterion(out, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

activations = {
    'ReLU':      nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(0.01),
    'GELU':      nn.GELU(),
    'Swish':     nn.SiLU(),
    'Mish':      nn.Mish(),
}

results = {}
EPOCHS = 3

for name, act in activations.items():
    torch.manual_seed(42)  # 동일 초기화
    model = CNN_with_act(act).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    history = {'train_acc': [], 'test_acc': [], 'test_loss': []}
    print(f"\n🚀 {name} 학습 중...")
    for epoch in range(EPOCHS):
        tl, ta = train_one_epoch(model, train_loader, opt, criterion)
        vl, va = evaluate(model, test_loader, criterion)
        history['train_acc'].append(ta)
        history['test_acc'].append(va)
        history['test_loss'].append(vl)
        print(f"   Epoch {epoch+1}: train_acc={ta:.4f}, test_acc={va:.4f}")

    results[name] = history

print("\n✅ 모든 활성화 함수 학습 완료")


In [ ]:
# 결과 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, EPOCHS+1)

for name, hist in results.items():
    axes[0].plot(epochs, hist['test_acc'], 'o-', label=name, linewidth=2, markersize=8)
    axes[1].plot(epochs, hist['test_loss'], 's-', label=name, linewidth=2, markersize=8)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('활성화 함수별 테스트 정확도')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Loss')
axes[1].set_title('활성화 함수별 테스트 손실')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# 최종 정확도 표
print("\n🏆 최종 테스트 정확도 (3 epoch)")
print("-" * 35)
for name, hist in sorted(results.items(), key=lambda x: -x[1]['test_acc'][-1]):
    print(f"   {name:>12}: {hist['test_acc'][-1]:.4f}")


## ✏️ 실습 과제 4-A

**문제:** Mish 함수를 NumPy로 직접 구현하고, PyTorch의 `F.mish`와 결과가 일치하는지 확인하세요.

수식: $\text{Mish}(x) = x \cdot \tanh(\ln(1 + e^x))$

> 💡 힌트: 큰 양수 입력에서 $e^x$가 오버플로될 수 있습니다. `np.log1p(np.exp(x))` 대신 안정적인 구현(softplus)을 사용하세요.

In [ ]:
# TODO: Numerical stability를 고려한 Mish 구현
def my_mish(x):
    # softplus(x) = log(1 + exp(x)) = max(x, 0) + log(1 + exp(-|x|))
    # softplus = np.maximum(x, 0) + np.log1p(np.exp(-np.abs(x)))
    # return x * np.tanh(softplus)
    pass

# 검증
# x_test = np.linspace(-100, 100, 201)
# our_result = my_mish(x_test)
# torch_result = F.mish(torch.tensor(x_test, dtype=torch.float32)).numpy()
# print(f"일치? {np.allclose(our_result, torch_result, atol=1e-5)}")


<details>
<summary>🔑 정답 보기</summary>

```python
def my_mish(x):
    # 안정적인 softplus 구현
    softplus = np.maximum(x, 0) + np.log1p(np.exp(-np.abs(x)))
    return x * np.tanh(softplus)
```
</details>

---
# 🎯 미니 프로젝트: 종합 실습

## 과제: 직접 만든 CNN 블록의 출력 크기 추적

다음 네트워크를 직접 만들고, 입력이 (1, 3, 224, 224)일 때 각 층의 출력 크기를 예측하고 검증하세요.

```
Conv 64ch  k=7 s=2 p=3  → ?
MaxPool    k=3 s=2 p=1  → ?
Conv 64ch  k=3 s=1 p=1  → ?
Conv 128ch k=3 s=2 p=1  → ?
GELU
Conv 256ch k=3 s=1 p=1  → ?
GAP                     → ?
FC 10                   → ?
```

In [ ]:
# 미니 프로젝트 - 여러분의 코드
class MyCustomNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 위 구조에 맞게 layer 정의
        # self.conv1 = ...
        pass

    def forward(self, x):
        # TODO: forward 작성하면서 각 단계 shape 출력
        # print(f"입력: {x.shape}")
        # x = self.conv1(x); print(f"Conv1 후: {x.shape}")
        # ...
        return x

# model = MyCustomNet().to(device)
# x = torch.randn(1, 3, 224, 224).to(device)
# out = model(x)
# print(f"\n최종 출력: {out.shape}")


<details>
<summary>🔑 정답 보기</summary>

```python
class MyCustomNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3)
        self.pool1 = nn.MaxPool2d(3, 2, 1)
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv3 = nn.Conv2d(64, 128, 3, 2, 1)
        self.gelu = nn.GELU()
        self.conv4 = nn.Conv2d(128, 256, 3, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, 10)

    def forward(self, x):
        x = self.conv1(x)   # (1, 64, 112, 112)
        x = self.pool1(x)   # (1, 64, 56, 56)
        x = self.conv2(x)   # (1, 64, 56, 56)
        x = self.conv3(x)   # (1, 128, 28, 28)
        x = self.gelu(x)
        x = self.conv4(x)   # (1, 256, 28, 28)
        x = self.gap(x)     # (1, 256, 1, 1)
        x = x.flatten(1)    # (1, 256)
        x = self.fc(x)      # (1, 10)
        return x
```
</details>

---
# 📝 1주차 1-2 마무리

## ✅ 핵심 정리

### 1. Convolution 연산
- 본질: 필터 슬라이딩 + 원소별 곱 + 합
- 구현: 이중 for문 → im2col 행렬 곱셈 (10배+ 빠름)
- 필터 = 학습 가능한 특징 검출기

### 2. Stride · Padding · Dilation
- **출력 크기 공식**: $O = \lfloor (W - K_{\text{eff}} + 2P) / S \rfloor + 1$
- **Same padding**: $P = (K-1)/2$ when $S=1, D=1$
- **Dilation**: 파라미터 추가 없이 receptive field 확장

### 3. Pooling
- Max Pool: 강한 특징 보존
- Avg Pool: 부드러운 다운샘플링
- **GAP**: FC 파라미터 폭발 방지, 최신 분류기의 표준
- **Receptive Field**: $RF_l = RF_{l-1} + (K_l-1) \cdot \prod S_i$

### 4. Activation Functions
- **ReLU**: 빠르지만 Dying 문제
- **GELU**: 트랜스포머/ViT 표준
- **Swish**: EfficientNet 표준
- **Mish**: YOLOv4 표준
- 트렌드: ReLU → GELU/Swish (부드러운 그래디언트)

## 📚 다음 차시 예고

**1-3. BatchNorm · Dropout · 학습 기법** (4시간)
- Batch Normalization vs Layer Normalization 원리
- Dropout 정규화 메커니즘
- Adam·SGD 최적화 비교
- 학습률 스케줄러 (Cosine, OneCycle)
- PyTorch 통합 학습 파이프라인

## 🔗 추천 자료

- 📖 [CS231n - Convolutional Networks](https://cs231n.github.io/convolutional-networks/)
- 📖 [A guide to convolution arithmetic](https://arxiv.org/abs/1603.07285) (Stride/Padding 시각화 명저)
- 📖 [Searching for Activation Functions (Swish 논문)](https://arxiv.org/abs/1710.05941)
- 📖 [GELU 논문](https://arxiv.org/abs/1606.08415)
- 🎥 [3Blue1Brown - Convolutions](https://www.youtube.com/watch?v=KuXjwB4LzSA)

---

**수고하셨습니다! 🎉**
